# Addestramento SFCN - Dataset IXI (T2) con 5-Fold Cross Validation & Bias Correction
Questo notebook implementa la pipeline completa per il dataset IXI con K-Fold.

**Design dello Split Stratificato**:
1. **20% Test Set**: Incontaminato, usato solo alla fine per la valutazione finale.
2. **20% Regression Val Set**: Usato *esclusivamente* a fine training (sul miglior modello) per fittare la retta della Bias Correction.
3. **60% Training Pool**: Utilizzato per eseguire una **5-Fold Cross Validation**. In ogni Fold, l'80% di questo pool fa da Train e il 20% da Validation (solo per l'Early Stopping). Al termine delle 5 Fold, si seleziona il modello assoluto migliore.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LinearRegression

from dp_model.model_files.sfcn import SFCN
from train import train_model
from dp_model import dp_utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Configurazione Parametri

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/collab4444/dataset-2-t1-t2/Prep_IXI_T2/Prep_IXI_T2"
CSV_PATH = "F:\\test\\ixi_info.csv"  # Aggiorna se esegui su Kaggle con dataset montato altrove!

# --- IMPOSTAZIONI K-FOLD E TRAINING ---
K_FOLDS = 5
OUTPUT_DIM = 100
EPOCHS = 130
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 15

# --- IMPOSTAZIONI TRANSFER LEARNING ---
USE_FINETUNING = True
FREEZE_EARLY_LAYERS = False  # False = Full Fine-Tuning (Consigliato T1->T2)

PRETRAINED_URL = "https://raw.githubusercontent.com/ha-ha-ha-han/UKBiobank_deep_pretrain/master/brain_age/run_20190719_00_epoch_best_mae.p"
PRETRAINED_PATH = "/kaggle/working/sfcn_official_pretrained.p"

## 2. Definizione del Dataset IXI (Standard, Senza Fallback Corrotti)

In [ ]:
class IXIBrainAgeDataset(Dataset):
    def __init__(self, data_dir, csv_path, is_train=False):
        self.data_dir = data_dir
        self.is_train = is_train
        self.samples = []
        
        self.bin_range = [0, OUTPUT_DIM]
        self.bin_step = 1
        self.sigma = 1.0
        
        df = pd.read_csv(csv_path)
        reference_date = datetime(2015, 2, 23)
        
        subfolders = ['Prep_Guys_T2', 'Prep_HH_T2', 'Prep_IOP_T2']
        
        for sf in subfolders:
            folder_path = os.path.join(data_dir, sf)
            if not os.path.exists(folder_path):
                continue
                
            for file in os.listdir(folder_path):
                if file.startswith("registered_image_") and file.endswith(".nii"):
                    nii_path = os.path.join(folder_path, file)
                    
                    ixi_id_str = file.replace("registered_image_", "").replace(".nii", "")
                    try:
                        ixi_id = int(ixi_id_str)
                    except ValueError:
                        continue
                        
                    row = df[df['IXI_ID'] == ixi_id]
                    if len(row) == 0:
                        continue
                        
                    dob_str = row.iloc[0]['DOB']
                    if pd.isna(dob_str):
                        continue
                        
                    try:
                        dob_str = str(dob_str).split(" ")[0]
                        dob = datetime.strptime(dob_str, "%Y-%m-%d")
                        true_age = (reference_date - dob).days / 365.25
                        
                        y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                        
                        self.samples.append({
                            "nii_path": nii_path,
                            "label_vect": y,
                            "true_age": true_age
                        })
                    except Exception as e:
                        continue
                        
        print(f"[{'TRAIN' if is_train else 'VAL/TEST'}] Caricati {len(self.samples)} pazienti IXI validi.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            dx = np.random.randint(-3, 4)
            dy = np.random.randint(-3, 4)
            dz = np.random.randint(-3, 4)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        
        if self.is_train and np.random.rand() > 0.5:
            data = np.flip(data, axis=0).copy()
            
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 3. Split Principale (60% Train Pool, 20% Reg Val, 20% Test)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

dummy_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    
    strat_classes_all = [int(a/5) for a in all_ages]
    
    # SPLIT 1: Separiamo il 20% per il TEST SET FINALE
    temp_idx, test_idx, temp_classes, _ = train_test_split(
        all_indices, strat_classes_all, 
        test_size=0.20, random_state=42, stratify=strat_classes_all
    )
    
    # SPLIT 2: Dal restante 80%, estraiamo 1/4 (cioè il 20% del totale originale) per la REGRESSIONE
    train_pool_idx, reg_val_idx = train_test_split(
        temp_idx, 
        test_size=0.25, random_state=42, stratify=temp_classes
    )
    
    print(f"\nSuddivisione Globale Completata:")
    print(f"- Pool K-Fold Training (60%): {len(train_pool_idx)} pazienti")
    print(f"- Regression Validation Set (20%): {len(reg_val_idx)} pazienti (solo per fittare Bias)")
    print(f"- Test Set Finale (20%): {len(test_idx)} pazienti")
    
    # Creazione dataset fissi per Regressione e Test
    reg_val_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
    test_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
    
    reg_val_dataset = torch.utils.data.Subset(reg_val_dataset, reg_val_idx)
    test_dataset = torch.utils.data.Subset(test_dataset, test_idx)
    
    reg_val_loader = DataLoader(reg_val_dataset, batch_size=1, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
else:
    print("ERRORE: Impossibile trovare i file. Controlla i percorsi.")

## 4. K-Fold Cross Validation sul Training Pool (60%)

In [ ]:
def load_pretrained_sfcn(output_dim, device, use_finetuning=True, freeze_early_layers=False):
    model = SFCN(output_dim=output_dim)
    if use_finetuning:
        if not os.path.exists(PRETRAINED_PATH):
            print("Scaricamento dei pesi ufficiali SFCN (UK Biobank)...")
            urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
            
        pretrained_state = torch.load(PRETRAINED_PATH, map_location='cpu')
        clean_state = {}
        for k, v in pretrained_state.items():
            name = k.replace("module.", "") if k.startswith("module.") else k
            clean_state[name] = v
            
        # Rimuove il classificatore finale
        filtered_state = {k: v for k, v in clean_state.items() if not k.startswith('classifier.conv_6')}
        model.load_state_dict(filtered_state, strict=False)
        
        if freeze_early_layers:
            for name, param in model.named_parameters():
                if any(name.startswith(f"feature_extractor.conv_{i}") for i in range(4)):
                    param.requires_grad = False
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    return model

best_models_paths = []
fold_maes = []

if dataset_size > 0:
    print("\n==============================================")
    print(f" INIZIO {K_FOLDS}-FOLD CV SUL TRAINING POOL")
    print("==============================================")
    
    train_pool_ages = [all_ages[i] for i in train_pool_idx]
    train_pool_classes = [int(a/5) for a in train_pool_ages]
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    
    for fold, (train_inner_idx, val_inner_idx) in enumerate(skf.split(train_pool_idx, train_pool_classes)):
        print(f"\n--- Esecuzione FOLD {fold+1}/{K_FOLDS} ---")
        
        fold_train_idx = [train_pool_idx[i] for i in train_inner_idx]
        fold_val_idx = [train_pool_idx[i] for i in val_inner_idx]
        
        fold_train_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=True)
        fold_val_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)
        
        fold_train_dataset = torch.utils.data.Subset(fold_train_dataset, fold_train_idx)
        fold_val_dataset = torch.utils.data.Subset(fold_val_dataset, fold_val_idx)
        
        fold_train_ages = [all_ages[i] for i in fold_train_idx]
        fold_train_classes = [int(a/5) for a in fold_train_ages]
        class_counts = np.bincount(fold_train_classes)
        weights = [1.0 / class_counts[c] if class_counts[c] > 0 else 0 for c in fold_train_classes]
        sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
        
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
        fold_val_loader = DataLoader(fold_val_dataset, batch_size=1, shuffle=False, num_workers=2)
        
        model = load_pretrained_sfcn(OUTPUT_DIM, device, USE_FINETUNING, FREEZE_EARLY_LAYERS)
        
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.Adam(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
        steps_per_epoch = len(fold_train_loader)
        
        trained_model, _, _, v_maes = train_model(
            model=model,
            train_loader=fold_train_loader,
            val_loader=fold_val_loader,
            optimizer=optimizer,
            device=device,
            epochs=EPOCHS,
            step_size=steps_per_epoch * 30,
            gamma=0.3,
            patience=PATIENCE
        )
        
        best_fold_mae = min(v_maes)
        fold_maes.append(best_fold_mae)
        
        model_save_path = os.path.join(MODELS_DIR, f"sfcn_IXI_T2_fold_{fold+1}.pth")
        if isinstance(trained_model, nn.DataParallel):
            torch.save(trained_model.module.state_dict(), model_save_path)
        else:
            torch.save(trained_model.state_dict(), model_save_path)
        
        best_models_paths.append(model_save_path)
        print(f"\n>>> FOLD {fold+1} COMPLETATA | MAE: {best_fold_mae:.3f} | Modello salvato: {model_save_path}")

## 5. Selezione Modello Assoluto e Age Bias Correction (sul 20% Reg Val)

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(" SELEZIONE BEST MODEL E BIAS CORRECTION FIT")
    print("==============================================")
    
    best_fold_idx = np.argmin(fold_maes)
    best_model_path = best_models_paths[best_fold_idx]
    print(f"Il miglior modello della K-Fold è il FOLD {best_fold_idx+1} (MAE: {fold_maes[best_fold_idx]:.3f})")
    print(f"Caricamento pesi da: {best_model_path}")
    
    best_model = SFCN(output_dim=OUTPUT_DIM)
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    best_model = best_model.to(device)
    best_model.eval()
    
    bin_centers = np.arange(0, OUTPUT_DIM, 1)
    
    # --- FIT DELLA REGRESSIONE SUL REGRESSION VALIDATION SET (20%) ---
    reg_true_ages, reg_preds = [], []
    
    with torch.no_grad():
        for inputs, _, true_age in reg_val_loader:
            inputs = inputs.to(device)
            out = best_model(inputs)[0].view(1, -1)
            prob = torch.exp(out).cpu().numpy()
            
            pred_age = (prob @ bin_centers)[0]
            
            reg_true_ages.append(true_age.item())
            reg_preds.append(pred_age)
            
    bias_model = LinearRegression()
    bias_model.fit(np.array(reg_preds).reshape(-1, 1), np.array(reg_true_ages))
    alpha = bias_model.coef_[0]
    beta = bias_model.intercept_
    print(f"\nFormula Bias Correction fittata sul Reg Val Set (20%):\nEtà_Corretta = {alpha:.3f} * Età_Predetta + {beta:.3f}")

## 6. Test Set Finale e Valutazione (sul 20% Test)

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(" VALUTAZIONE FINALE SUL TEST SET INCONTAMINATO")
    print("==============================================")
    
    test_true_ages, test_preds = [], []
    
    with torch.no_grad():
        for inputs, _, true_age in test_loader:
            inputs = inputs.to(device)
            out = best_model(inputs)[0].view(1, -1)
            prob = torch.exp(out).cpu().numpy()
            
            pred_age = (prob @ bin_centers)[0]
            
            test_true_ages.append(true_age.item())
            test_preds.append(pred_age)
            
    test_preds_corrected = bias_model.predict(np.array(test_preds).reshape(-1, 1))
    
    test_true_ages = np.array(test_true_ages)
    test_preds = np.array(test_preds)
    
    mae_pre = np.mean(np.abs(test_preds - test_true_ages))
    mae_post = np.mean(np.abs(test_preds_corrected - test_true_ages))
    
    print(f"\n>>> MAE SUL TEST SET (Pre-Correzione) : {mae_pre:.3f} Anni")
    print(f">>> MAE SUL TEST SET (Post-Correzione): {mae_post:.3f} Anni")
    
    plt.figure(figsize=(9, 7))
    plt.scatter(test_true_ages, test_preds, color='silver', edgecolor='gray', alpha=0.6, s=50, label='Pre-Correzione')
    plt.scatter(test_true_ages, test_preds_corrected, color='forestgreen', edgecolor='black', alpha=0.9, s=90, marker='*', label='Post-Correzione')
    
    min_val = min(min(test_true_ages), min(test_preds), min(test_preds_corrected)) - 2
    max_val = max(max(test_true_ages), max(test_preds), max(test_preds_corrected)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Ideale')
    
    plt.title(f'Valutazione IXI Finale (Miglior Fold)\nMAE Finale: {mae_post:.3f} Anni\n(Correzione: Età = {alpha:.3f}*Pred + {beta:.3f})', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Età Reale (Anni)', fontsize=12)
    plt.ylabel('Età Predetta (Anni)', fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '09_ixi_final_test_results.png'), dpi=300)
    plt.show()
